# Protótipo e testes — `ResidualHistograms`

Notebook autocontido para prototipar e testar a classe `ResidualHistograms`.

A classe recebe dados no formato final do notebook do 3W:

`(n_windows, window_size, n_signals)`

e os nomes dos sinais na mesma ordem do terceiro eixo. Para cada sinal, calcula `ground_truth - prediction`, gera um histograma individual, fornece estatísticas e permite mostrar ou salvar os gráficos.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


## 2. Dados simulados

Os dados seguem o exemplo real do Windowing: 1839 janelas, 128 pontos e 17 sinais.


In [ ]:
N_WINDOWS = 1839
WINDOW_SIZE = 128

SIGNAL_NAMES = [
    "ESTADO-DHSV", "ESTADO-M1", "ESTADO-M2", "ESTADO-PXO",
    "ESTADO-SDV-GL", "ESTADO-SDV-P", "ESTADO-W1", "ESTADO-W2",
    "ESTADO-XO", "P-ANULAR", "P-JUS-CKGL", "P-MON-CKP",
    "P-PDG", "P-TPT", "QGL", "T-JUS-CKP", "T-TPT",
]

N_VARIABLES = len(SIGNAL_NAMES)
rng = np.random.default_rng(42)

ground_truth = rng.normal(
    0.0, 1.0,
    size=(N_WINDOWS, WINDOW_SIZE, N_VARIABLES)
).astype(np.float32)

noise_std = np.linspace(0.05, 0.30, N_VARIABLES).astype(np.float32)

noise = (
    rng.normal(size=ground_truth.shape)
    * noise_std.reshape(1, 1, -1)
).astype(np.float32)

prediction = ground_truth + noise

print("Ground truth:", ground_truth.shape)
print("Prediction:  ", prediction.shape)
print("Sinais:      ", N_VARIABLES)

assert ground_truth.shape == (N_WINDOWS, WINDOW_SIZE, N_VARIABLES)
assert prediction.shape == ground_truth.shape
assert np.isfinite(ground_truth).all()
assert np.isfinite(prediction).all()

print("[OK] Dados simulados válidos.")


## 3. Protótipo da classe

In [ ]:
class ResidualHistograms:
    """Calcula e visualiza histogramas dos resíduos multissinais."""

    def __init__(self, ground_truth, prediction, signal_names):
        self.ground_truth = np.asarray(ground_truth, dtype=np.float32)
        self.prediction = np.asarray(prediction, dtype=np.float32)
        self.signal_names = list(signal_names)

        self._validate_inputs()
        self.residuals = self._calculate_residuals()

    def _validate_inputs(self):
        if self.ground_truth.ndim != 3:
            raise ValueError(
                "ground_truth deve ter shape "
                "(n_windows, window_size, n_signals)."
            )

        if self.prediction.ndim != 3:
            raise ValueError(
                "prediction deve ter shape "
                "(n_windows, window_size, n_signals)."
            )

        if self.ground_truth.shape != self.prediction.shape:
            raise ValueError(
                "ground_truth e prediction devem possuir o mesmo shape. "
                f"Recebidos: {self.ground_truth.shape} e "
                f"{self.prediction.shape}."
            )

        if len(self.signal_names) != self.ground_truth.shape[2]:
            raise ValueError(
                "O número de signal_names deve ser igual ao número de sinais."
            )

        if len(set(self.signal_names)) != len(self.signal_names):
            raise ValueError("signal_names contém nomes duplicados.")

        if not np.isfinite(self.ground_truth).all():
            raise ValueError("ground_truth contém NaN ou Inf.")

        if not np.isfinite(self.prediction).all():
            raise ValueError("prediction contém NaN ou Inf.")

    def _calculate_residuals(self):
        return (self.ground_truth - self.prediction).astype(np.float32)

    def _get_signal_index(self, signal):
        if isinstance(signal, (int, np.integer)):
            if signal < 0 or signal >= len(self.signal_names):
                raise IndexError(f"Índice de sinal inválido: {signal}.")
            return int(signal)

        if signal not in self.signal_names:
            raise KeyError(f"Sinal não encontrado: {signal!r}.")

        return self.signal_names.index(signal)

    def get_signal_names(self):
        return self.signal_names.copy()

    def get_signal_residuals(self, signal):
        index = self._get_signal_index(signal)
        return self.residuals[:, :, index].ravel().copy()

    def statistics(self, signal):
        residuals = self.get_signal_residuals(signal)
        return pd.Series(residuals, name=str(signal)).describe()

    def all_statistics(self):
        rows = []

        for signal in self.signal_names:
            stats = self.statistics(signal)
            rows.append({
                "signal": signal,
                "count": stats["count"],
                "mean": stats["mean"],
                "std": stats["std"],
                "min": stats["min"],
                "25%": stats["25%"],
                "50%": stats["50%"],
                "75%": stats["75%"],
                "max": stats["max"],
            })

        return pd.DataFrame(rows).set_index("signal")

    def _sanitize_filename(self, signal):
        sanitized = str(signal)
        for char in '<>:"/\\|?*':
            sanitized = sanitized.replace(char, "_")
        return sanitized

    def plot_signal(
        self,
        signal,
        bins=50,
        figsize=(10, 5),
        show=True,
        save_path=None,
    ):
        residuals = self.get_signal_residuals(signal)

        plt.figure(figsize=figsize)
        plt.hist(residuals, bins=bins)

        plt.axvline(0, linestyle="--", linewidth=1)
        plt.xlabel("Resíduo (real - previsão)")
        plt.ylabel("Frequência")
        plt.title(f"Histograma dos resíduos — {signal}")
        plt.grid(alpha=0.2)
        plt.tight_layout()

        if save_path is not None:
            save_path = Path(save_path)
            save_path.mkdir(parents=True, exist_ok=True)

            filename = self._sanitize_filename(signal) + ".png"
            plt.savefig(
                save_path / filename,
                bbox_inches="tight",
            )

        if show:
            plt.show()
        else:
            plt.close()

    def plot_all(
        self,
        bins=50,
        figsize=(10, 5),
        show=True,
        save_path=None,
    ):
        for signal in self.signal_names:
            self.plot_signal(
                signal=signal,
                bins=bins,
                figsize=figsize,
                show=show,
                save_path=save_path,
            )


## 4. Instanciar a classe

In [ ]:
error_analyzer = ResidualHistograms(
    ground_truth=ground_truth,
    prediction=prediction,
    signal_names=SIGNAL_NAMES,
)

print("[OK] ResidualHistograms criado.")


## 5. Teste do shape e dos valores dos resíduos

In [ ]:
residuals = error_analyzer.residuals

print("Shape dos resíduos:", residuals.shape)

assert residuals.shape == ground_truth.shape
assert residuals.ndim == 3
assert np.isfinite(residuals).all()

print("[OK] Resíduos válidos.")


## 6. Teste do cálculo dos resíduos

In [ ]:
expected = ground_truth - prediction

assert np.allclose(
    error_analyzer.residuals,
    expected,
)

print("[OK] Cálculo ground_truth - prediction validado.")


## 7. Teste de previsão perfeita

In [ ]:
perfect_test = ResidualHistograms(
    ground_truth=ground_truth,
    prediction=ground_truth.copy(),
    signal_names=SIGNAL_NAMES,
)

assert np.allclose(
    perfect_test.residuals,
    0.0,
)

print("[OK] Previsão perfeita produz resíduos = 0.")


## 8. Teste de resíduos de um sinal

In [ ]:
signal = "P-PDG"

signal_residuals = error_analyzer.get_signal_residuals(signal)

expected_count = N_WINDOWS * WINDOW_SIZE

print("Sinal:", signal)
print("Shape:", signal_residuals.shape)
print("Número de resíduos:", signal_residuals.size)

assert signal_residuals.shape == (expected_count,)
assert np.isfinite(signal_residuals).all()

print("[OK] 1839 × 128 resíduos encontrados para o sinal.")


## 9. Teste de acesso por nome e índice

In [ ]:
by_name = error_analyzer.get_signal_residuals("P-PDG")
by_index = error_analyzer.get_signal_residuals(
    SIGNAL_NAMES.index("P-PDG")
)

assert np.array_equal(by_name, by_index)

print("[OK] Acesso por nome e índice validado.")


## 10. Listar sinais

In [ ]:
available_signals = error_analyzer.get_signal_names()

print("Sinais disponíveis:")
for signal in available_signals:
    print("-", signal)

assert available_signals == SIGNAL_NAMES

print("[OK] Nomes preservados na ordem correta.")


## 11. Estatísticas de um sinal

In [ ]:
stats = error_analyzer.statistics("P-PDG")
display(stats)

assert np.isfinite(stats.to_numpy()).all()

print("[OK] Estatísticas calculadas.")


## 12. Estatísticas de todos os sinais

In [ ]:
all_stats = error_analyzer.all_statistics()
display(all_stats)

assert list(all_stats.index) == SIGNAL_NAMES
assert all_stats.shape == (N_VARIABLES, 8)

print("[OK] Estatísticas de todos os sinais calculadas.")


## 13. Histograma individual — uso em notebook

In [ ]:
error_analyzer.plot_signal(
    signal="P-PDG",
    bins=50,
    show=True,
)


## 14. Teste de salvamento

In [ ]:
output_dir = Path("test_residual_histograms")

error_analyzer.plot_signal(
    signal="P-PDG",
    bins=50,
    show=False,
    save_path=output_dir,
)

expected_file = output_dir / "P-PDG.png"

print("Arquivo:", expected_file)
print("Existe:", expected_file.exists())

assert expected_file.is_file()

print("[OK] Histograma salvo com o nome do sinal.")


## 15. Gerar e salvar todos os histogramas

In [ ]:
all_output_dir = Path("test_residual_histograms_all")

error_analyzer.plot_all(
    bins=50,
    show=False,
    save_path=all_output_dir,
)

saved_files = sorted(all_output_dir.glob("*.png"))

print("Arquivos salvos:", len(saved_files))
for file in saved_files:
    print("-", file.name)

assert len(saved_files) == N_VARIABLES

saved_names = sorted(file.stem for file in saved_files)
expected_names = sorted(SIGNAL_NAMES)

assert saved_names == expected_names

print("[OK] Todos os histogramas foram salvos corretamente.")


## 16. Testes de validação — entradas inválidas

In [ ]:
def assert_raises_value_error(func, description):
    try:
        func()
    except ValueError:
        print(f"[OK] {description}")
    else:
        raise AssertionError(
            f"Esperava ValueError: {description}"
        )


assert_raises_value_error(
    lambda: ResidualHistograms(
        ground_truth[0],
        prediction[0],
        SIGNAL_NAMES,
    ),
    "entrada não 3D",
)

assert_raises_value_error(
    lambda: ResidualHistograms(
        ground_truth,
        prediction[:, :, :-1],
        SIGNAL_NAMES,
    ),
    "shapes diferentes",
)

assert_raises_value_error(
    lambda: ResidualHistograms(
        ground_truth,
        prediction,
        SIGNAL_NAMES[:-1],
    ),
    "quantidade incorreta de nomes",
)

duplicate_names = SIGNAL_NAMES.copy()
duplicate_names[-1] = duplicate_names[0]

assert_raises_value_error(
    lambda: ResidualHistograms(
        ground_truth,
        prediction,
        duplicate_names,
    ),
    "nomes duplicados",
)

print("[OK] Validações estruturais passaram.")


## 17. Teste explícito de NaN e Inf

In [ ]:
ground_truth_nan = ground_truth.copy()
ground_truth_nan[0, 0, 0] = np.nan

assert_raises_value_error(
    lambda: ResidualHistograms(
        ground_truth_nan,
        prediction,
        SIGNAL_NAMES,
    ),
    "ground_truth contendo NaN",
)

prediction_inf = prediction.copy()
prediction_inf[0, 0, 0] = np.inf

assert_raises_value_error(
    lambda: ResidualHistograms(
        ground_truth,
        prediction_inf,
        SIGNAL_NAMES,
    ),
    "prediction contendo Inf",
)


## 18. Teste de sanitização de nomes de arquivo

In [ ]:
special_names = ["SIGNAL/ONE", "SIGNAL:TWO"]

small_ground_truth = np.zeros(
    (2, 4, 2),
    dtype=np.float32,
)

small_prediction = np.ones(
    (2, 4, 2),
    dtype=np.float32,
)

special_test = ResidualHistograms(
    ground_truth=small_ground_truth,
    prediction=small_prediction,
    signal_names=special_names,
)

special_output_dir = Path(
    "test_residual_histograms_special_names"
)

special_test.plot_all(
    show=False,
    save_path=special_output_dir,
)

special_files = sorted(
    special_output_dir.glob("*.png")
)

for file in special_files:
    print("-", file.name)

assert len(special_files) == 2
assert {file.name for file in special_files} == {
    "SIGNAL_ONE.png",
    "SIGNAL_TWO.png",
}

print("[OK] Nomes de arquivo sanitizados.")


## 19. Validação final

Principais requisitos testados:

- entrada multissinal 3D;
- shapes compatíveis;
- resíduos ponto a ponto;
- um conjunto de resíduos por sinal;
- acesso por nome;
- estatísticas;
- histograma individual;
- salvamento;
- geração de todos os histogramas;
- rejeição de NaN/Inf;
- rejeição de nomes inválidos/duplicados;
- sanitização dos nomes dos arquivos.


In [ ]:
assert error_analyzer.ground_truth.shape == (
    N_WINDOWS, WINDOW_SIZE, N_VARIABLES
)

assert error_analyzer.prediction.shape == (
    N_WINDOWS, WINDOW_SIZE, N_VARIABLES
)

assert error_analyzer.residuals.shape == (
    N_WINDOWS, WINDOW_SIZE, N_VARIABLES
)

for signal in SIGNAL_NAMES:
    values = error_analyzer.get_signal_residuals(signal)

    assert values.shape == (
        N_WINDOWS * WINDOW_SIZE,
    )

    assert np.isfinite(values).all()

assert len(list(all_output_dir.glob("*.png"))) == N_VARIABLES

print("==============================================")
print(" VALIDAÇÃO FINAL")
print("==============================================")
print(f"Número de janelas: {N_WINDOWS}")
print(f"Window size:       {WINDOW_SIZE}")
print(f"Variáveis:         {N_VARIABLES}")
print(f"Resíduos por sinal: {N_WINDOWS * WINDOW_SIZE}")
print(f"Histogramas:        {N_VARIABLES}")
print("==============================================")
print(" PROTÓTIPO: PASSOU")
print("==============================================")


## 20. Próximo passo

Depois de validar o comportamento deste protótipo, a implementação definitiva pode ser extraída para o projeto principal.

Antes disso, vale revisar a API final — especialmente se `statistics()`, `all_statistics()` e acesso por índice devem permanecer na classe.
